# 03 — Fitur Statis: GloVe & FastText

**Tujuan:** mengubah tiap soal menjadi satu vektor 300 dimensi dengan *mean pooling* dari embedding kata pretrained, dan mengukur **OOV (out-of-vocabulary)** untuk menjawab **RQ2** (apakah subword FastText mengatasi OOV lebih baik daripada GloVe).

**Input:** `data_clean.csv` (kolom `text_clean`, dari Notebook 01)

**Output** (semua baris sejajar dengan `data_clean.csv`, terurut menurut `id`):
- `features/glove.npy` — matriks N × 300
- `features/fasttext.npy` — matriks N × 300
- `tables/oov_stats.csv`, `tables/glove_oov_top.csv`, `tables/fasttext_wordvocab_oov_top.csv`
- `features/static_meta.json` — catatan embedding dan pooling untuk Bab III-D

**Embedding:**
| | File | Ukuran | Sifat |
|---|---|---|---|
| GloVe | `glove.6B.300d.txt` | 400 ribu kata | level kata, tanpa subword. Kata di luar vocab diabaikan |
| FastText | `cc.en.300.bin` | 2 juta kata + n-gram karakter | kata di luar vocab tetap mendapat vektor dari subword |

**Catatan penting**
- Embedding dan pooling **tidak memakai label dan tidak di-fit pada data**, jadi seluruh 3.256 soal boleh diproses sekaligus tanpa risiko kebocoran. Split baru dipakai di Notebook 05-06.
- Dijalankan di **Kaggle** dengan **Internet ON** jika file embedding belum ada. FastText `.bin` sekitar 7 GB (unduhan 4,5 GB), jadi butuh RAM besar dan waktu unduh beberapa menit.
- Alternatif tanpa unduh: tambahkan Kaggle Dataset berisi `glove.6B.300d.txt` dan `cc.en.300.bin` lewat *Add Data*. Notebook ini mencari file berdasarkan nama di `/kaggle/input`.

## 1. Import & Konfigurasi

In [1]:
import glob
import gzip
import json
import os
import shutil
import subprocess
import zipfile
from collections import Counter

import numpy as np
import pandas as pd

ON_KAGGLE = os.path.exists('/kaggle/input')
OUT_DIR = '/kaggle/working' if ON_KAGGLE else '.'
TEMP_DIR = '/kaggle/temp' if os.path.exists('/kaggle/temp') else '.'
FEAT_DIR = os.path.join(OUT_DIR, 'features')
TAB_DIR = os.path.join(OUT_DIR, 'tables')
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

GLOVE_NAME = 'glove.6B.300d.txt'
GLOVE_URL = 'https://nlp.stanford.edu/data/glove.6B.zip'
FT_NAME = 'cc.en.300.bin'
FT_URL = 'https://dl.fbaipublicfiles.com/fasttext/vectors-english/cc.en.300.bin.gz'
DIM = 300


def find_file(name):
    # Cari file menurut nama: folder kerja, TEMP_DIR, lalu semua Kaggle Dataset yang ditambahkan
    for p in [name, os.path.join(TEMP_DIR, name)]:
        if os.path.exists(p):
            return p
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return hits[0] if hits else None


def download(url, dest):
    print('Mengunduh:', url)
    subprocess.run(['wget', '-q', '-c', '-O', dest, url], check=True)

## 2. Load Data

In [2]:
DATA_PATH = find_file('data_clean.csv')
assert DATA_PATH is not None, 'data_clean.csv tidak ditemukan (tambahkan Kaggle Dataset-nya)'
df = pd.read_csv(DATA_PATH)
assert df['id'].is_monotonic_increasing and df['id'].is_unique, 'urutan harus menurut id'
assert df['text_clean'].notna().all()

docs = df['text_clean'].str.split().tolist()
tok_counter = Counter(t for d in docs for t in d)
vocab = set(tok_counter)

print('Jumlah soal         :', len(df))
print('Total token         :', sum(tok_counter.values()))
print('Vocab dataset (tipe):', len(vocab))

Jumlah soal         : 3256
Total token         : 265031
Vocab dataset (tipe): 8253


## 3. GloVe

Hanya baris GloVe untuk kata yang ada di dataset yang dibaca ke memori (lebih hemat dan lebih cepat daripada memuat seluruh 400 ribu kata). Hasil OOV tidak berubah karena kata yang tidak ada di dataset memang tidak berpengaruh.

In [3]:
glove_path = find_file(GLOVE_NAME)
if glove_path is None:
    zip_path = os.path.join(TEMP_DIR, 'glove.6B.zip')
    download(GLOVE_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extract(GLOVE_NAME, TEMP_DIR)
    os.remove(zip_path)
    glove_path = os.path.join(TEMP_DIR, GLOVE_NAME)
print('GloVe:', glove_path)

glove_vec = {}
with open(glove_path, encoding='utf-8') as f:
    for line in f:
        word, rest = line.split(' ', 1)
        if word in vocab:
            glove_vec[word] = np.asarray(rest.split(), dtype=np.float32)

assert all(v.shape == (DIM,) for v in glove_vec.values()), 'dimensi GloVe tidak 300'
print('Tipe dataset yang ada di GloVe:', len(glove_vec), 'dari', len(vocab))

GloVe: /kaggle/input/datasets/thanakomsn/glove6b300dtxt/glove.6B.300d.txt
Tipe dataset yang ada di GloVe: 6227 dari 8253


In [4]:
def pool_mean(doc, lookup):
    # Rata-rata vektor kata yang punya vektor; dokumen tanpa satu pun kata berukuran nol vektor nol
    vecs = [lookup[t] for t in doc if t in lookup]
    return (np.mean(vecs, axis=0) if vecs else np.zeros(DIM, dtype=np.float32)), len(vecs)


glove_X = np.zeros((len(docs), DIM), dtype=np.float32)
glove_n_in = np.zeros(len(docs), dtype=int)
for i, d in enumerate(docs):
    glove_X[i], glove_n_in[i] = pool_mean(d, glove_vec)

n_tok = np.array([len(d) for d in docs])
glove_doc_oov = 1 - glove_n_in / n_tok

glove_oov_types = [t for t in vocab if t not in glove_vec]
glove_stats = {
    'token_oov_rate_%': round(100 * sum(tok_counter[t] for t in glove_oov_types) / sum(tok_counter.values()), 2),
    'type_oov_rate_%': round(100 * len(glove_oov_types) / len(vocab), 2),
    'docs_all_oov': int((glove_n_in == 0).sum()),
    'doc_oov_mean_%': round(100 * glove_doc_oov.mean(), 2),
    'doc_oov_max_%': round(100 * glove_doc_oov.max(), 2),
}
print(json.dumps(glove_stats, indent=2))

glove_top = pd.DataFrame(sorted(((t, tok_counter[t]) for t in glove_oov_types), key=lambda x: -x[1])[:50],
                         columns=['token', 'frequency'])
glove_top.to_csv(os.path.join(TAB_DIR, 'glove_oov_top.csv'), index=False)
glove_top.head(20)

{
  "token_oov_rate_%": 7.52,
  "type_oov_rate_%": 24.55,
  "docs_all_oov": 0,
  "doc_oov_mean_%": 8.07,
  "doc_oov_max_%": 72.73
}


,token,frequency
0,nums,9969
1,subarray,902
2,subarrays,431
3,leetcode,193
4,varchar,110
5,starttime,70
6,getrandom,32
7,settimeout,31
8,getvalue,31
9,undergroundsystem,30


## 4. FastText

Dimuat dengan `gensim.models.fasttext.load_facebook_vectors`, yang menyertakan n-gram karakter. Kata yang tidak ada di vocab kata tetap mendapat vektor dari subword-nya, jadi tidak ada kata yang dilewati.

Untuk membandingkan dengan GloVe, "OOV" di FastText didefinisikan sebagai **kata yang tidak ada di vocab kata** (ia tetap diberi vektor lewat subword). Itulah kata-kata yang diselamatkan oleh subword.

In [5]:
ft_path = find_file(FT_NAME)
if ft_path is None:
    gz_path = os.path.join(TEMP_DIR, FT_NAME + '.gz')
    download(FT_URL, gz_path)
    ft_path = os.path.join(TEMP_DIR, FT_NAME)
    with gzip.open(gz_path, 'rb') as fin, open(ft_path, 'wb') as fout:
        shutil.copyfileobj(fin, fout)
    os.remove(gz_path)
print('FastText:', ft_path)

from gensim.models.fasttext import load_facebook_vectors
ft = load_facebook_vectors(ft_path)
assert ft.vector_size == DIM, 'dimensi FastText tidak 300'
print('Ukuran vocab kata FastText:', len(ft.key_to_index))

FastText: /kaggle/input/datasets/sanyatargrenkin/cc-en-300-bin/cc.en.300.bin
Ukuran vocab kata FastText: 2000000


In [6]:
# Hitung vektor tiap tipe kata sekali saja (lebih cepat daripada per token)
ft_vec = {t: ft.get_vector(t).astype(np.float32) for t in vocab}
ft_in_vocab = {t: (t in ft.key_to_index) for t in vocab}

ft_X = np.zeros((len(docs), DIM), dtype=np.float32)
ft_n_wordvocab = np.zeros(len(docs), dtype=int)
for i, d in enumerate(docs):
    ft_X[i] = np.mean([ft_vec[t] for t in d], axis=0)
    ft_n_wordvocab[i] = sum(ft_in_vocab[t] for t in d)

ft_doc_oov = 1 - ft_n_wordvocab / n_tok
ft_oov_types = [t for t in vocab if not ft_in_vocab[t]]
ft_stats = {
    'token_oov_rate_%': round(100 * sum(tok_counter[t] for t in ft_oov_types) / sum(tok_counter.values()), 2),
    'type_oov_rate_%': round(100 * len(ft_oov_types) / len(vocab), 2),
    'docs_all_oov': int((ft_n_wordvocab == 0).sum()),
    'doc_oov_mean_%': round(100 * ft_doc_oov.mean(), 2),
    'doc_oov_max_%': round(100 * ft_doc_oov.max(), 2),
}
print(json.dumps(ft_stats, indent=2))

ft_top = pd.DataFrame(sorted(((t, tok_counter[t]) for t in ft_oov_types), key=lambda x: -x[1])[:50],
                      columns=['token', 'frequency'])
ft_top.to_csv(os.path.join(TAB_DIR, 'fasttext_wordvocab_oov_top.csv'), index=False)
ft_top.head(20)

{
  "token_oov_rate_%": 2.79,
  "type_oov_rate_%": 21.71,
  "docs_all_oov": 0,
  "doc_oov_mean_%": 2.32,
  "doc_oov_max_%": 53.97
}


,token,frequency
0,leetcode,193
1,getrandom,32
2,settimeout,31
3,getvalue,31
4,undergroundsystem,30
5,skiplist,29
6,hasnext,29
7,workertimes,28
8,firstevent,26
9,beginword,25


## 5. Perbandingan OOV (bahan RQ2)

Catatan interpretasi: pada GloVe, kata OOV **hilang** dari rata-rata; pada FastText, kata di luar vocab kata tetap **dipertahankan** lewat subword. Jadi selisih angka OOV bukan selisih kualitas, tetapi selisih informasi yang terbuang. Kualitas klasifikasi diukur di Notebook 05-06.

In [7]:
oov_table = pd.DataFrame({'GloVe': glove_stats, 'FastText (word vocab)': ft_stats})
oov_table.to_csv(os.path.join(TAB_DIR, 'oov_stats.csv'))
oov_table

,GloVe,FastText (word vocab)
token_oov_rate_%,7.52,2.79
type_oov_rate_%,24.55,21.71
docs_all_oov,0.00,0.00
doc_oov_mean_%,8.07,2.32
doc_oov_max_%,72.73,53.97


## 6. Sanity Check

In [8]:
for name, X in [('glove', glove_X), ('fasttext', ft_X)]:
    assert X.shape == (len(df), DIM), f'{name}: bentuk salah {X.shape}'
    assert np.isfinite(X).all(), f'{name}: ada NaN/inf'
    n_zero = int((np.linalg.norm(X, axis=1) == 0).sum())
    print(f'{name:9s} shape={X.shape}  baris nol={n_zero}  norm rata-rata={np.linalg.norm(X, axis=1).mean():.3f}')


def top_similar(X, idx, k=5):
    Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)
    sims = Xn @ Xn[idx]
    order = np.argsort(-sims)
    return [(df.loc[j, 'title'], round(float(sims[j]), 3)) for j in order if j != idx][:k]


idx = int(df.index[df['id'] == 1][0]) if (df['id'] == 1).any() else 0
print()
print('Soal acuan:', df.loc[idx, 'title'])
print('GloVe   :', top_similar(glove_X, idx))
print('FastText:', top_similar(ft_X, idx))

glove     shape=(3256, 300)  baris nol=0  norm rata-rata=2.980
fasttext  shape=(3256, 300)  baris nol=0  norm rata-rata=0.589

Soal acuan: Two Sum
GloVe   : [('Combination Sum IV', 0.939), ('Minimum Number of Operations to Make Arrays Similar', 0.935), ('Minimum Size Subarray Sum', 0.928), ('Apply Bitwise Operations to Make Strings Equal', 0.927), ('Even Number of Knight Moves', 0.912)]
FastText: [('Minimum Number of Operations to Make Arrays Similar', 0.964), ('Find the Duplicate Number', 0.958), ('Combination Sum IV', 0.957), ('Minimum Increments for Target Multiples in an Array', 0.956), ('Minimum Operations to Reach Target Array', 0.955)]


## 7. Simpan

In [9]:
np.save(os.path.join(FEAT_DIR, 'glove.npy'), glove_X)
np.save(os.path.join(FEAT_DIR, 'fasttext.npy'), ft_X)

meta = {
    'n_rows': int(len(df)),
    'dim': DIM,
    'text_view': 'text_clean (lowercase, letters only, stopwords removed)',
    'pooling': 'mean over token vectors; GloVe skips OOV tokens (zero vector if none), FastText uses subword vectors for all tokens',
    'glove_file': GLOVE_NAME,
    'fasttext_file': FT_NAME,
    'oov': {'glove': glove_stats, 'fasttext_word_vocab': ft_stats},
}
with open(os.path.join(FEAT_DIR, 'static_meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

# Verifikasi baca ulang
assert np.load(os.path.join(FEAT_DIR, 'glove.npy')).shape == (len(df), DIM)
assert np.load(os.path.join(FEAT_DIR, 'fasttext.npy')).shape == (len(df), DIM)
print('Tersimpan di', FEAT_DIR)

Tersimpan di /kaggle/working/features


## Ringkasan untuk paper (Bab III-D dan RQ2)

Salin dari output:
- token OOV rate dan type OOV rate GloVe vs FastText (tabel `oov_stats.csv`)
- contoh token OOV paling sering (`glove_oov_top.csv`): biasanya singkatan atau istilah teknis. Pilih 5-10 sebagai contoh di pembahasan
- jumlah soal yang vektornya nol di GloVe (`docs_all_oov`)

Setelah file `.npy` ada, unggah folder `features/` sebagai Kaggle Dataset (private) agar Notebook 05-06 tidak perlu mengulang ekstraksi.